# Step 2.12: mixed fidelity and the fidelity check

This notebook is the check of step 2.12. A backtest now records the data each trade date filled on, its fidelity era: TBBO over trades over 1-second bars over 1-minute bars, the finest each date holds (`docs/masterplan/02-engine-and-data.md` §1.6). A run whose dates include trades or 1-second bars also reruns the stretch those dates span, from flat, at its own fidelity, at 1-second, and at 1-minute fidelity, and keeps the differences in trades, net profit and loss, maximum drawdown, and fill prices: its fidelity check, the error bar of its dates on bars. `run.report()` prints the eras, the main metrics with the trades split by the fidelity of their entry, and the check.

The notebook generates a year of ES sessions with 1-minute bars throughout and trades and 1-second bars for its last quarter, backtests a strategy over it, and prints the run's report. If your store holds MES's 1-minute bars, it then prints the report of a run on them, where every day runs at 1-minute fidelity and no check is possible. The year is generated by seed into your `FEFAT_HOME` cache folder and may be deleted at any time. Pick the `.venv` kernel and run all cells.

## A year with ticks for its last quarter

The year is ESH2026's sessions from 2025-03-24 to 2026-03-19, the trade dates of CME's equity index calendar, so none on a holiday, each as dense as the committed fixtures and with three planted quiet stretches and three price jumps, as the year of step 2.7 has them. `scripts/gen_fixtures.py` writes the 1-minute bars of every session, and the trades and 1-second bars of the last quarter of the sessions alone, into `<FEFAT_HOME>/cache/synthetic-es-2025-03-24-2026-03-19/`. Generating it takes a few minutes the first time; later the cell reuses it.

Expected on this PC, the first time: `249 sessions from 2025-03-24 to 2026-03-19, ticks from 2025-12-17, ready in 80 s`, give or take; later runs print the same line with a time of 0 s.

In [ ]:
import os
import subprocess
import sys
import time
from contextlib import contextmanager
from datetime import date, timedelta
from pathlib import Path

import polars as pl

import fefat as ff
from fefat.backtest import FidelityCheck, RunConfig
from fefat.config import resolve_paths
from fefat.market.calendar import counted
from fefat.market.saved_instruments import instrument_table

REPOSITORY = Path(ff.__file__).resolve().parents[4]
FIRST, LAST = date(2025, 3, 24), date(2026, 3, 19)
paths = resolve_paths(os.environ)
table = instrument_table(paths.data_dir)
calendar = table.sessions_of("ES", paths.cache_dir)
sessions = calendar.between(FIRST, LAST).get_column("trade_date").to_list()
sessions = sessions[-int(os.environ.get("FEFAT_CHECK_DAYS", len(sessions))) :]
TICKS_FROM = sessions[len(sessions) * 3 // 4]
year = paths.cache_dir / f"synthetic-es-{sessions[0]}-{LAST}"
year_store = year / "data"
started = time.perf_counter()
if not (year_store / "synthetic" / "MANIFEST.json").is_file():
    generator = REPOSITORY / "scripts" / "gen_fixtures.py"
    dates = ["--dates", f"{sessions[0]}..{LAST}", "--contracts", "ESH2026"]
    ticks = ["--ticks-from", str(TICKS_FROM), "--planted-gaps", "3"]
    command = [
        sys.executable,
        str(generator),
        "--out",
        str(year_store / "synthetic"),
        *dates,
        *ticks,
    ]
    generated = subprocess.run(command, capture_output=True, text=True, check=False)
    if generated.returncode != 0:
        raise RuntimeError(f"Expected gen_fixtures.py to pass, got: {generated.stderr}")
print(
    f"{counted(len(sessions), 'session')} from {sessions[0]} to {LAST}, ticks from "
    f"{TICKS_FROM}, ready in {time.perf_counter() - started:.0f} s"
)

## The run and its report

A long on each 5-minute close above the highest of the 12 closes before it, with a stop of 12 ticks and a target of 24, one position at a time, at the default settings: each trade date fills on its finest data, one tick of slippage on market and stop orders, and the fees of `retail-default`, whose first schedule starts after the year, so a warning names it (F-FEE-01). The run reads the year's store, while `FEFAT_DATA_DIR` points there for the cell alone, and keeps its record and its engine cache under your `FEFAT_HOME`.

Expected on this PC: `Ran in 4 s`, then the report. Its fidelity eras are `1m` from 2025-03-25, the year's second session, the first whose front the roll calendar decides, to 2025-12-16, 185 trade dates, then `trades` from 2025-12-17 to 2026-03-19, 63 trade dates. Then come the trades by the fidelity of their entry, the metrics, and the fidelity check of the last quarter: the baseline on trades, then the reruns at `1s` and `1m`, each with the data its trade dates filled on, its trades, net P&L, and maximum drawdown, the differences from the baseline in brackets, and how far its fill prices lie from the baseline's, in ticks. The generated tape is a random walk, so the strategy loses, and its drawdown passes the capital: here the eras and the check matter, not the strategy.

In [ ]:
STOP_TICKS = 12


def breakout_on(root):
    """A long on a 5-minute close above the highest of the 12 closes before it, with a stop of
    `STOP_TICKS` and a target twice as far."""
    return ff.SignalStrategy(
        instrument=root,
        timeframe="5m",
        features={"highest": pl.col("close").rolling_max(12).shift(1)},
        long_entry=pl.col("close") > pl.col("highest"),
        short_entry=pl.lit(False),
        execution=ff.Execution(
            entry=ff.Market(),
            stop=ff.Stop(ticks=STOP_TICKS),
            targets=[ff.Target(ticks=2 * STOP_TICKS)],
            sizing=ff.Fixed(1),
        ),
    )


@contextmanager
def store_at(folder):
    """`FEFAT_DATA_DIR` pointed at `folder` while the block runs, then back at yours."""
    yours = os.environ.get("FEFAT_DATA_DIR")
    os.environ["FEFAT_DATA_DIR"] = str(folder)
    try:
        yield
    finally:
        if yours is None:
            os.environ.pop("FEFAT_DATA_DIR")
        else:
            os.environ["FEFAT_DATA_DIR"] = yours


started = time.perf_counter()
with store_at(year_store):
    run = ff.backtest.run(breakout_on("ES"), sessions[0], LAST)
print(f"Ran in {time.perf_counter() - started:.0f} s\n")
run.report()

## Reading the deltas

`ff.backtest.fidelity_check(run)` works the check out again from the saved run, on the data it read, and gives the check its metrics hold. Below, each rerun's net P&L difference as a share of the baseline's, and the stop beside the typical 1-minute range of the check's days.

Expected: `True`, then `At 1s: net P&L +637.50 from the baseline's -32,999.68, 1.9% of its size`, `At 1m: net P&L +6,275.00 from the baseline's -32,999.68, 19.0% of its size`, and `The stop lies 12 ticks from the entry; a 1-minute bar of the check's days spans 2 ticks at the median.` The 1-second fills lie within the few percent the step calls typical, while the 1-minute fills come out 19 % better: a planted price jump inside a bar lets the bar fill a stop at its price, where the trades fill it at the first print past the jump, up to 52 ticks away. That is the error bar of the year's 1-minute days, and the stop is not inside the typical 1-minute range, so it is the gaps, not the stop, that the bars miss.

In [ ]:
with store_at(year_store):
    CHECK = ff.backtest.fidelity_check(run)
same = run.metrics.fidelity_check == CHECK
print(f"fidelity_check(run) gives the check run.metrics holds: {same}")
if not isinstance(CHECK, FidelityCheck):
    raise ValueError(f"Expected a fidelity check of the year's last quarter, got {CHECK!r}")
baseline = CHECK.baseline
for rerun in filter(None, (CHECK.at_one_second, CHECK.at_one_minute)):
    share = "" if not baseline.pnl else f", {abs(rerun.pnl_delta / baseline.pnl):.1%} of its size"
    print(
        f"At {rerun.outcome.fidelity}: net P&L {rerun.pnl_delta:+,.2f} from the baseline's "
        f"{baseline.pnl:,.2f}{share}"
    )
tick = int(table.get("ES").tick_size.scaleb(9))
minutes = pl.scan_parquet((year_store / "synthetic" / "ohlcv-1m").as_posix() + "/**/*.parquet")
spans = minutes.filter(pl.col("trade_date").is_between(CHECK.first, CHECK.last))
median = spans.select(((pl.col("high") - pl.col("low")) / tick).median()).collect().item()
print(
    f"The stop lies {STOP_TICKS} ticks from the entry; a 1-minute bar of the check's days spans "
    f"{median:.0f} ticks at the median."
)

## A run on your MES

The same strategy over the last 30 days of your MES's 1-minute bars, from the provider that holds them. Your store holds no trades or 1-second bars of MES, so every day runs at 1-minute fidelity and the run has no fidelity check: its report says so, and names the file a check needs.

Expected: a report whose fidelity eras are a single `1m` era, ending with `No fidelity check: every trade date ran at 1-minute fidelity`. Without MES in your store, one line says so.

In [ ]:
listed = ff.market.datasets().filter(pl.col("root") == "MES", pl.col("schema") == "ohlcv-1m")
if listed.is_empty():
    print(
        "No MES 1-minute bars in your store: import them (docs/data-guide.md) to see a run on "
        "1-minute bars alone."
    )
else:
    mes = listed.row(0, named=True)
    config = RunConfig(provider=mes["provider"])
    last = mes["last_trade_date"]
    ff.backtest.run(breakout_on("MES"), last - timedelta(days=30), last, config).report()

## Your check

Run `fidelity_check` on a real strategy over your tick year: read the deltas; a few percent of net P&L between tick and 1-second fills is typical, more than that tells you the strategy is sensitive to fills.

No tick year is on this PC yet (the step's amendment of 2026-10-21), so the generated year stands in for it above, and the MES report shows what a run on 1-minute bars alone says until you import a trades or 1-second file. If the 1-minute rerun differs wildly, check that the strategy's stops are not inside the typical 1-minute range, as the last line of the reading above compares them: that is a property of the strategy, not a bug.